# 06 - Deep Learning SITS Classification (TempCNN / Transformer)

Upgrades the Random Forest classifier of notebook 02 to **deep temporal models** that learn the full Rabi phenology from the Sentinel-1/2 time-series. Includes a **Transformer** with self-attention and a **TempCNN**, plus a **Prithvi foundation-model** fine-tuning head.

> Requires `torch`. Train on GPU for the full 8-state dataset. Here we demonstrate the training loop on a synthetic SITS tensor that mirrors real (batch, time, channels) shape.

In [1]:
import sys
sys.path.append('..')
import numpy as np, torch
from torch.utils.data import TensorDataset, DataLoader
from src import deep_models

T, C = 12, 8   # 12 timesteps (fortnights), 8 bands (VV,VH + 6 optical)
torch.manual_seed(0); np.random.seed(0)

## Build training tensors
In production, extract per-pixel (or per-parcel) time-series from the GEE feature stack of notebook 02 and label them with ground-truth points. Here we synthesise a separable two-class SITS so the architecture and training loop are verified end-to-end.

In [2]:
def make_sits(n, T, C):
    t = np.linspace(0, 1, T)
    X, y = [], []
    for _ in range(n):
        cls = np.random.randint(2)
        base = np.random.normal(0, 0.05, (T, C))
        if cls == 1:  # wheat: strong mid-season green-up in optical bands
            curve = np.exp(-((t - 0.5) ** 2) / 0.02)
            base[:, 2:] += curve[:, None] * np.random.uniform(0.5, 0.9)
            base[:, 1] += curve * 0.4  # VH rises with biomass
        X.append(base); y.append(cls)
    return np.array(X, np.float32), np.array(y, np.int64)

Xtr, ytr = make_sits(2000, T, C); Xte, yte = make_sits(500, T, C)
tr = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), batch_size=64, shuffle=True)
te = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte)), batch_size=128)
Xtr.shape

(2000, 12, 8)

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = deep_models.TransformerSITS(n_channels=C, n_classes=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = torch.nn.CrossEntropyLoss()
for epoch in range(15):
    loss = deep_models.train_epoch(model, tr, opt, crit, device)
    if (epoch + 1) % 5 == 0:
        acc = deep_models.evaluate(model, te, device)
        print(f'epoch {epoch+1:02d} | loss {loss:.4f} | test acc {acc:.3f}')

epoch 05 | loss 0.0009 | test acc 1.000
epoch 10 | loss 0.0004 | test acc 1.000
epoch 15 | loss 0.0003 | test acc 1.000


In [4]:
# TempCNN baseline for comparison
cnn = deep_models.TempCNN(n_channels=C, n_classes=2, seq_len=T).to(device)
opt = torch.optim.AdamW(cnn.parameters(), lr=1e-3)
for epoch in range(15):
    deep_models.train_epoch(cnn, tr, opt, crit, device)
print('TempCNN test acc:', round(deep_models.evaluate(cnn, te, device), 3))

TempCNN test acc: 1.0


## Prithvi foundation-model fine-tuning (outline)
Load frozen Prithvi-100M, extract embeddings for HLS chips, train only `PrithviHead`. This transfers global geospatial priors to wheat mapping with very few labels.

In [5]:
# from huggingface_hub import hf_hub_download  # pip install huggingface_hub timm
# backbone = load_prithvi(...)  # frozen ViT encoder over HLS 6-band chips
# emb = backbone(chips)         # (B, 768)
head = deep_models.PrithviHead(embed_dim=768, n_classes=2).to(device)
demo_emb = torch.randn(4, 768, device=device)
print('Prithvi head output:', head(demo_emb).shape)  # (4, 2)

Prithvi head output: torch.Size([4, 2])


In [6]:
# Persist the best model for inference / GEE export of probabilities
torch.save(model.state_dict(), '../outputs/transformer_sits.pt')
print('Saved transformer_sits.pt')

Saved transformer_sits.pt
